#### Summary:
In this notebook I'll prepare the inputs for both modality (RNA, ATAC) associations, metadata inputs, etc. Then further notebooks will perform the associations and necessary meta-analyses.

Required inputs:

Final Seurat object with metadata columns for donor ID (library here) and cell type (major_celltypes_fin here)

In [1]:
suppressMessages(library(hdf5r)) 
suppressMessages(library(Seurat))
suppressMessages(library(DESeq2))

suppressMessages(library(dplyr)) 
suppressMessages(library(ggplot2))
suppressMessages(library(ggpubr)) 

suppressMessages(library(Matrix)) 
suppressMessages(library(data.table))
suppressMessages(library(future)) 
suppressMessages(library(stringr))
suppressMessages(library(stringi))
suppressPackageStartupMessages(library(parallel))
suppressPackageStartupMessages(library(readr))

suppressMessages(library(enrichR))
suppressMessages(library(fgsea))
suppressMessages(library(ggrepel))
suppressMessages(library(RColorBrewer))
suppressMessages(library(shadowtext))
suppressMessages(library(forcats))

# Basic Inputs

In [2]:
overall_outdir <- '/dir/to/save/outputs/to'
outdir <- file.path(overall_outdir,'trait_associations')

In [3]:
#read in donor metadata so we can get lists of samples per study
meta_fp <- file.path(overall_outdir, 'all_datasets_donor_metadata3.tsv')
meta <- read.table(meta_fp, sep='\t', header=1)
head(meta)

alberta_samples <- subset(meta, dataset=='Alberta') %>% pull(sample)
hpap_rna_samples <- subset(meta, dataset=='HPAP' & grepl('RNA',assays)) %>% pull(sample)
hpap_atac_samples <- subset(meta, dataset=='HPAP' & grepl('ATAC',assays)) %>% pull(sample)
wang_samples2 <- subset(meta, dataset=='Wang' & assays=='snATAC') %>% pull(sample)
wang_samples2

,sample,age,sex,BMI,HbA1c,dataset,culture_time,TSSe,tissue_source,chemistry,assays,race_ancestry_ethnicity,purity,cause_of_death
,<chr>,<int>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>
1,R207,50,Female,22.2,NA,Alberta,70,3.450389,Alberta,Multiome,Multiome,NA,90,NA
2,R217,71,Female,35.5,6.3,Alberta,15,3.530508,Alberta,Multiome,Multiome,NA,80,NA
3,R218,73,Female,28.4,5.9,Alberta,70,3.641667,Alberta,Multiome,Multiome,NA,80,NA
4,R221,44,Male,30.5,5.3,Alberta,136,3.792649,Alberta,Multiome,Multiome,NA,75,NA
5,R223,54,Male,27.0,5.8,Alberta,64,2.992320,Alberta,Multiome,Multiome,NA,90,NA
6,R226,30,Female,32.3,4.9,Alberta,16,3.864731,Alberta,Multiome,Multiome,NA,95,NA


[1] "JYH792" "MM110"  "MM123"  "MM124"  "MM56"   "MM59"   "MM80"   "MM86"  
 [9] "MM89"   "MM95"   "MM98"

In [4]:
celltypes <- c('beta', 'alpha', 'delta', 'gamma', 'acinar','ductal', 'endothelial', 'immune','stellate')

# 1. Read in Seurat objects and map to common cell types

### Alberta

In [11]:
#read in the adata object 
rds_fp = '/path/to/Alberta/multiome/final_object.rds'
adata1 = readRDS(rds_fp)
adata1

Loading required package: Signac



An object of class Seurat 
571775 features across 174819 samples within 4 assays 
Active assay: ATAC (210485 features, 210485 variable features)
 3 other assays present: RNA, SCT, ATAC_CTpeaks
 7 dimensional reductions calculated: pca, harmony.rna, umap.rna, lsi, harmony.atac, umap.atac, umap.wnn

In [13]:
table(adata1$major_celltypes_fin)


     acinar       alpha        beta       delta      ductal endothelial 
      27387       44500       81100        8911        5452         342 
      gamma      immune    stellate 
       5504         633         990 

### HPAP RNA

In [14]:
#read in hpap object
rds_fp2 <- '/path/to/HPAP/RNA/final_object.rds'
adata2 <- readRDS(rds_fp2)
adata2

An object of class Seurat 
36601 features across 192203 samples within 1 assay 
Active assay: RNA (36601 features, 2000 variable features)
 3 dimensional reductions calculated: pca, harmony, umap

In [15]:
# subset down to the ND samples
adata2_nd <- subset(adata2, subset=Diabetes_Status_w_AAB=='ND')
adata2_nd

An object of class Seurat 
36601 features across 71871 samples within 1 assay 
Active assay: RNA (36601 features, 2000 variable features)
 3 dimensional reductions calculated: pca, harmony, umap

In [25]:
adata2 <- NULL

In [27]:
#rename celltypes to match the alberta ones -- combine mast and macrophage into immune
celltype_map <- unique(adata2_nd$cell_type)
new_celltypes <- c('alpha','alpha+beta','stellate','stellate','endothelial','delta',
                   'beta','gamma','immune', 'ductal','acinar','alpha','ductal','immune')
names(celltype_map) <- new_celltypes
adata2_nd$major_celltypes_fin <- plyr::mapvalues(adata2_nd$cell_type, from=celltype_map, to=names(celltype_map))
table(adata2_nd$cell_type, adata2_nd$major_celltypes_fin)

                    
                     acinar alpha alpha+beta  beta delta ductal endothelial
  Acinar              14419     0          0     0     0      0           0
  Active_Stellate         0     0          0     0     0      0           0
  Alpha                   0 21197          0     0     0      0           0
  Alpha+Beta              0     0       1819     0     0      0           0
  Beta                    0     0          0 17671     0      0           0
  Cycling_Alpha           0   279          0     0     0      0           0
  Delta                   0     0          0     0  1993      0           0
  Ductal                  0     0          0     0     0   5461           0
  Endothelial             0     0          0     0     0      0        2695
  Gamma+Epsilon           0     0          0     0     0      0           0
  Macrophage              0     0          0     0     0      0           0
  Mast                    0     0          0     0     0      0    

### HPAP ATAC

In [16]:
#read in hpap object
rds_fp3 <- '/path/to/HPAP/ATAC/final_object.rds'
adata3 <- readRDS(rds_fp3)
adata3

An object of class Seurat 
1303997 features across 97837 samples within 4 assays 
Active assay: ATAC_peaks (481311 features, 481311 variable features)
 3 other assays present: RNA, Final_Peaks, Unified_Peaks
 3 dimensional reductions calculated: lsi, harmony.atac, umap.atac

In [17]:
# subset down to the ND samples
adata3_nd <- subset(adata3, subset=condition=='Control')
adata3_nd

An object of class Seurat 
1303997 features across 66872 samples within 4 assays 
Active assay: ATAC_peaks (481311 features, 481311 variable features)
 3 other assays present: RNA, Final_Peaks, Unified_Peaks
 3 dimensional reductions calculated: lsi, harmony.atac, umap.atac

In [28]:
adata3 <- NULL
gc()

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,17432110,931.0,33630378,1796.1,19880362,1061.8
Vcells,31831059602,242851.8,47988084268,366120.1,42096769948,321172.9


In [29]:
#rename celltypes to match the alberta ones -- combine mast and macrophage into immune
celltype_map <- unique(adata3_nd$Cell.Type)
new_celltypes <- c('acinar','ductal','alpha','beta','delta','immune','gamma',
                   'stellate','stellate','stellate','endothelial','ductal')
names(celltype_map) <- new_celltypes
adata3_nd$major_celltypes_fin <- plyr::mapvalues(adata3_nd$Cell.Type, from=celltype_map, to=names(celltype_map))
table(adata3_nd$Cell.Type, adata3_nd$major_celltypes_fin)

              
               acinar alpha  beta delta ductal endothelial gamma immune
  A_Stellate        0     0     0     0      0           0     0      0
  Acinar         8189     0     0     0      0           0     0      0
  Alpha             0 23455     0     0      0           0     0      0
  Beta              0     0 20445     0      0           0     0      0
  Delta             0     0     0  4026      0           0     0      0
  Ductal            0     0     0     0   5927           0     0      0
  Endothelial       0     0     0     0      0        1169     0      0
  Gamma             0     0     0     0      0           0   997      0
  Immune            0     0     0     0      0           0     0    356
  MUC5B_Ductal      0     0     0     0    986           0     0      0
  Q_Stellate        0     0     0     0      0           0     0      0
  Schwann           0     0     0     0      0           0     0      0
              
               stellate
  A_Stella

### Wang Multiome

In [ ]:
#get rna and atac cell type numbers from Wang Seurat object
wang_rds_fp <- "/nfs/lab/projects/islet_multiomics_stress/data/rds/multiome.unionPeaks.WNN.qc_v3.rds"
adata6 <- readRDS(wang_rds_fp)
adata6

An object of class Seurat 
432141 features across 97387 samples within 4 assays 
Active assay: integrated.rpca (3000 features, 3000 variable features)
 3 other assays present: RNA, SCT, unionpeaks
 6 dimensional reductions calculated: pca, umap, lsi, atac.harmony, atac.umap, wnn.umap

In [ ]:
wang_map <- c('beta'='beta','alpha'='alpha','delta'='delta','gamma'='gamma','acinar'='acinar','ductal'='ductal','alpha_beta'='rm', 
              'a.stellate'='stellate','q.stellate'='stellate','endothelial'='endothelial','immune'='immune')
adata6$major_celltypes_fin <- plyr::mapvalues(adata6$atac.anno2, from=names(wang_map), to=wang_map)
table(adata6$atac.anno2,adata6$major_celltypes_fin)

             
              acinar alpha  beta delta ductal endothelial gamma immune    rm
  a.stellate       0     0     0     0      0           0     0      0     0
  acinar        5915     0     0     0      0           0     0      0     0
  alpha            0 27199     0     0      0           0     0      0     0
  alpha_beta       0     0     0     0      0           0     0      0 23662
  beta             0     0 26959     0      0           0     0      0     0
  delta            0     0     0  5508      0           0     0      0     0
  ductal           0     0     0     0   3627           0     0      0     0
  endothelial      0     0     0     0      0         332     0      0     0
  gamma            0     0     0     0      0           0  1943      0     0
  immune           0     0     0     0      0           0     0    452     0
  q.stellate       0     0     0     0      0           0     0      0     0
             
              stellate
  a.stellate       764
  

In [ ]:
#subset to only Wang samples and non rm cells
adata6_cut <- subset(adata6, subset=sample %in% wang_samples)
adata6_fin <- subset(adata6_cut, subset=major_celltypes_fin!='rm')
adata6_fin

Warning message:
"Keys should be one or more alphanumeric characters followed by an underscore, setting key from atac.umap_ to atacumap_"


An object of class Seurat 
432141 features across 13763 samples within 4 assays 
Active assay: integrated.rpca (3000 features, 3000 variable features)
 3 other assays present: RNA, SCT, unionpeaks
 6 dimensional reductions calculated: pca, umap, lsi, atac.harmony, atac.umap, wnn.umap

In [ ]:
adata6 <- NULL
adata6_cut <- NULL
gc()

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,15201268,811.9,24252770,1295.3,24036421,1283.7
Vcells,6567713305,50107.7,12121707034,92481.3,12103029386,92338.8


# 2. Prepare combined pseudobulk RNA counts tables for DESeq

In [112]:
### Function to sum up all counts for a cell type by sample
get_per_sample_gex_SUMS <- function(adata, gex.counts, sample_bcs, dataset, cell_type, samples, outdir){
    #pull out rows of gex.counts where BC Ident matches cell.type
    bcs <- names(Idents(adata)[Idents(adata) == cell_type])
    counts <- gex.counts[,colnames(gex.counts) %in% bcs]

    #initialize the matrix of sample gex
    counts.df <- as.data.frame(rep(0,length(row.names(gex.counts))))
    row.names(counts.df) <- row.names(gex.counts)
    colnames(counts.df) <- c('temp')

    #go through samples and calculate sum of gex values
    for (sample in samples){
        sample_cols <- colnames(counts) %in% sample_bcs[[sample]]
        counts.cut <- counts[,sample_cols]
        
        #if only one bc, this becomes a vector which is an issue
        if (typeof(counts.cut) == 'double'){
            mean.counts <- round(counts.cut)
        #if there are NO bcs, this will return NA (just return 0 for everything)
            } else if(length(colnames(counts.cut)) == 0){
            mean.counts <- rep(0,length(row.names(counts)))
            } else {
            mean.counts <- round(rowSums(counts.cut))
            }
        counts.df <- cbind(counts.df,as.data.frame(mean.counts))
         }
    fin.counts.df <- counts.df[,-c(1)]
    colnames(fin.counts.df) <- samples

    #export df
    mtx.fp <- file.path(outdir,sprintf('%s_%s_sample_gex_total_counts.txt',dataset, cell_type))
    write.table(fin.counts.df,mtx.fp,sep='\t',quote=FALSE)
}

In [109]:
### Wrapper to make RNA per donor counts matrices
wrapper_make_ct_rna_matrices <- function(adata, samples, sample_col, dataset, matrix_dir){
    # Get gex matrix
    Idents(adata) <- adata$major_celltypes_fin
    DefaultAssay(adata) <- 'RNA'
    gex.counts <- GetAssayData(adata, slot='counts')
    dim(gex.counts)
    
    # Get sample barcodes
    sample_bcs <- list()
    for (sample in samples){
        sample_bcs[[sample]] <- row.names(adata[[]][adata[[]][,sample_col] == sample,])
    }

    # Run matrix function for each cell type
    for (celltype in celltypes){
        get_per_sample_gex_SUMS(adata, gex.counts, sample_bcs, dataset, celltype, samples, matrix_dir)
    }
}

In [111]:
matrix_dir1 <- file.path(outdir, 'RNA', 'sample_matrices')
dir.create(matrix_dir1, showWarnings = FALSE)

In [113]:
wrapper_make_ct_rna_matrices(adata1, alberta_samples, 'library', 'Alberta', matrix_dir1)
wrapper_make_ct_rna_matrices(adata2_nd, hpap_rna_samples, 'library', 'HPAP', matrix_dir1)

In [132]:
#for each cell type read in all the matrices and combine into one! -- then delete dataset specific ones
for (celltype in celltypes){
    fps <- list.files(matrix_dir1, pattern=celltype)
    fin_fps <- fps[!grepl('all_datasets',fps)]
    dfs <- list()
    for(fp in fin_fps){
        df <- read.table(file.path(matrix_dir1, fp), sep='\t', header=1) %>% tibble::rownames_to_column(var='gene')
        dfs[[fp]] <- df
        system(sprintf('rm %s',file.path(matrix_dir1, fp)))
    }
    fin_mat <- left_join(dfs[[1]], dfs[[2]], by='gene') %>% left_join(dfs[[3]], by='gene') %>% left_join(dfs[[4]], by='gene')
    out_fp <- file.path(matrix_dir1, sprintf('%s_all_datasets_sample_gex_total_counts.tsv', celltype))
    write.table(fin_mat, out_fp, sep='\t', row.names=FALSE, quote=FALSE)
}

# 3. Run featureCounts to generate ATAC matrices

### Make featureCounts inputs: cell type labeled filtered barcode files

In [ ]:
#read back in final metadata df and gather sample lists
meta_fp <- file.path(outdir, 'all_datasets_donor_combined_metadata.tsv')
meta <- read.table(meta_fp, sep='\t', header=1)
hpap_atac_samples <- subset(meta, dataset=='HPAP' & grepl('ATAC',assays)) %>% pull(sample)
wang_samples <- subset(meta, dataset=='Wang' & assay!='Multiome') %>% pull(sample)

In [49]:
atac_dir <- file.path(outdir,'ATAC')
bc_dir <- file.path(atac_dir,'sample_barcodes')
celltypes

[1] "beta"        "alpha"       "delta"       "gamma"       "acinar"     
[6] "ductal"      "endothelial" "immune"      "stellate"

### HPAP dataset

In [ ]:
#generate files with the list of barcodes for each sample for cell types of interest
#with second column labeled by cell type (with sample name appended to it)
bc_outdir <- file.path(outdir, 'ATAC', 'hpap_sample_barcodes')

for (sample in samples){    
    print(sample)
    # pull out all barcodes for the sample
    sample_bcs = Cells(so)[so[[]]$library==sample]
    
    # make df with cell type info
    bc_df <- data.frame(bc=sample_bcs, celltype=paste(so[[]]
library==sample]))
    print(dim(bc_df))
    
    # cut down to celltypes of interest then add sample prefix to cell type name
    bc_df_cut <- subset(bc_df, celltype %in% joint_celltypes)
    bc_df_cut
celltype, sep='_')
    print(dim(bc_df_cut))
    
    # reformat bc names and write to file
    bc_df_cut
bc, 10, 27)
    
    sample_bc_fp = file.path(bc_outdir, sprintf('%s.filtered_barcodes_wCTs_ofInterest.txt',sample))
    write.table(bc_df_cut, sample_bc_fp, sep='\t', row.names=F, col.names=F, quote=F)
}

### Wang et al. dataset

In [ ]:
#first read in original donor metadata table to get which ones were ND
meta_fp <- '/path/to/Wang/info/Wang_2023_STable1a.csv'
meta <- fread(meta_fp)
wang_snatac_samples <- subset(meta, `Diabetes status`=='Non-diabetic' & `10x snATAC-seq assay`=='Yes')  %>% pull(Donor)
length(wang_snatac_samples)

#write this to a file
write(wang_snatac_samples, '/path/to/Wang/info/Wang_snATAC_samples.txt', sep='\n')

[1] 11

In [58]:
#now read in snATAC-object annotations supplemental table
wang_snatac_meta <- '/path/to/Wang/info/snATAC/GSE169453_cell_annotation.csv'
wang_meta <- read.table(wang_snatac_meta, sep=',', header=1)

#map leiden clusters to cell types using Figure 1b from paper
wang_ct_map <- c('alpha','alpha','alpha',
                 'beta','beta','beta','beta',
                 'delta','gamma','acinar','ductal',
                 'stellate','immune','endothelial')
names(wang_ct_map) <- c(0,4,8,1,2,3,5,6,10,7,9,11,12,13)
wang_meta$celltype <- plyr::mapvalues(wang_meta$leiden, from=names(wang_ct_map), to=wang_ct_map)

dataset <- 'Wang_snATAC'
for(sample in wang_snatac_samples){
    #get relevant info in correct format
    bc_df <- subset(wang_meta, donor==sample) %>% select(index, donor, celltype)
    bc_df$bc <- paste0(str_split_fixed(bc_df$index, '_', 2)[,2],'-1')
    bc_df$ct_id <- paste(bc_df$donor, bc_df$celltype, sep='_')
    bc_df <- select(bc_df, bc, ct_id)

    #now write this to a file
    sample_bc_fp = file.path(bc_dir, dataset, sprintf('%s.filtered_barcodes_cts.txt', sample))
    write.table(bc_df, sample_bc_fp, sep='\t', row.names=F, col.names=F, quote=F)
}

### Run featureCounts in the terminal

### Clean up featureCounts outputs by dataset

In [5]:
options(scipen=999)

In [6]:
### Function to process a featureCounts output matrix into an simplified format
### really just removing some columns and updating labels
convert_featureCounts_mtx <- function(fc_fp, celltype, out_fp){
    # Read in output df and make peaks row names
    df <- as.data.frame(vroom::vroom(fc_fp, delim='\t', skip=1))
    row.names(df) <- paste(df$Chr, df$Start-1, df$End, sep='-')
    #for whatever reason the saf file has the Start coordinate as one after the peak start, so need to 
    #subtract one here for this to match all other things made with the same peaks list
    
    # Remove unnecessary columns, then simplify colnames
    fin_df <- df[,-c(1,2,3,4,5,6)]
    new_names <- gsub(sprintf('_%s.bam', celltype), '', sapply(strsplit(colnames(fin_df), split='/'), tail, n=1))
    colnames(fin_df) <- new_names
    write.table(fin_df, out_fp, sep='\t', quote=F)
}

In [7]:
mtx_dir <- file.path(outdir,'ATAC','merged_matrices')

In [ ]:
dataset <- 'HPAP'
fc_dir <- file.path(outdir,'ATAC','feature_counts',dataset,'feature_counts')

for (celltype in celltypes){
    fc_fp <- file.path(fc_dir, sprintf('%s_featureCounts_mtx.txt', celltype))
    out_fp <- file.path(mtx_dir ,sprintf('%s_%s_sample_peak_total_counts.txt', dataset, celltype))
    convert_featureCounts_mtx(fc_fp, celltype, out_fp)
}

In [12]:
dataset <- 'Wang_snATAC'
fc_dir <- file.path(outdir,'ATAC','feature_counts',dataset,'feature_counts')

for (celltype in celltypes){
    fc_fp <- file.path(fc_dir, sprintf('%s_featureCounts_mtx.txt', celltype))
    out_fp <- file.path(mtx_dir ,sprintf('%s_%s_sample_peak_total_counts.txt', dataset, celltype))
    convert_featureCounts_mtx(fc_fp, celltype, out_fp)
}

Rows: 291821 Columns: 17
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr  (3): Geneid, Chr, Strand
dbl (14): Start, End, Length, /nfs/lab/projects/multiomic_islet/outputs/revi...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 291821 Columns: 17
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr  (3): Geneid, Chr, Strand
dbl (14): Start, End, Length, /nfs/lab/projects/multiomic_islet/outputs/revi...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 291821 Columns: 17
── Column s

# 5. Finalize merged ATAC matrices (all donors pseudobulked by cell type)

In [9]:
#gather file paths for all different datasets
prev_prefix <- '/path/to/Alberta/sample_matrices/'
prev_suffix <- '_sample_union_peaks_total_counts.txt'

hpap_snatac_prefix <- file.path(mtx_dir,'HPAP_snATAC_')
hpap_snatac_suffix <- '_sample_peak_total_counts.txt'

wang_snatac_prefix <- file.path(mtx_dir,'Wang_snATAC_')
wang_snatac_suffix <- '_sample_peak_total_counts.txt'

In [10]:
#read in union peaks so can standardize rows before merging dfs
union_peaks_fp <- '/path/to/peak/calls/union_peaks.sort.bed'
union_peaks_df <- read.table(union_peaks_fp, sep='\t')
union_peaks <- paste(union_peaks_df$V1, union_peaks_df$V2, union_peaks_df$V3, sep='-')
length(union_peaks)
head(union_peaks)

[1] 291821

[1] "chr1-9956-10256"    "chr1-29219-29482"   "chr1-99568-99868"  
[4] "chr1-102793-103093" "chr1-127658-127913" "chr1-180714-181014"

In [11]:
for(celltype in celltypes){
    print(celltype)
    
    #read in all matrices and make sure in order of sorted union peaks list
    df1 <- read.table(paste0(prev_prefix,celltype,prev_suffix), sep='\t', header=1)[union_peaks,]
    df2 <- read.table(paste0(hpap_multiome_prefix,celltype,hpap_multiome_suffix), sep='\t', header=1)[union_peaks,]
    df3 <- read.table(paste0(wang_snatac_prefix,celltype,wang_snatac_suffix), sep='\t', header=1)[union_peaks,]
    dfs <- list(df1, df2, df3)
    
    merged_df <- do.call("cbind",dfs)
    print(dim(merged_df))

    #write this to a file
    fin_outdir <- file.path(outdir,'ATAC','final_matrices')
    out_fp <- file.path(fin_outdir, sprintf('%s_all_donor_peak_total_counts.tsv',celltype))
    write.table(merged_df, out_fp, sep='\t', quote=F)
}

[1] "beta"
[1] 291821     87
[1] "alpha"
[1] 291821     87
[1] "delta"
[1] 291821     87
[1] "gamma"
[1] 291821     87
[1] "acinar"
[1] 291821     87
[1] "ductal"
[1] 291821     87
[1] "endothelial"
[1] 291821     87
[1] "immune"
[1] 291821     86
[1] "stellate"
[1] 291821     87
